# SQL: Дополнение 1 (Расширенные команды)

В этом ноутбуке мы разберем команды для более гибкого поиска и фильтрации данных.

## Содержание
1. [Подготовка окружения](#Подготовка-окружения)
2. [DISTINCT (Уникальные значения и работа с колонками)](#1.-DISTINCT)
3. [LIKE (Поиск по шаблону и работа с символами/числами)](#2.-LIKE)
4. [IN и BETWEEN (Фильтрация списков и диапазонов)](#3.-IN-и-BETWEEN)
5. [Дополнительные задачи](#Задачи)

In [59]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.executescript('''
CREATE TABLE Categories (id INTEGER PRIMARY KEY, name TEXT NOT NULL);
CREATE TABLE Products (id INTEGER PRIMARY KEY, name TEXT NOT NULL, price REAL, category_id INTEGER);
CREATE TABLE Orders (id INTEGER PRIMARY KEY, product_id INTEGER, quantity INTEGER, order_date TEXT);

INSERT INTO Categories (name) VALUES ('Электроника'), ('Одежда'), ('Книги'), ('Мебель');
INSERT INTO Products (name, price, category_id) VALUES 
    ('Смартфон', 50000, 1), ('Ноутбук', 80000, 1), ('Телевизор', 45000, 1),
    ('Футболка', 1500, 2), ('Джинсы', 3500, 2), ('Куртка', 7000, 2),
    ('SQL для начинающих', 1200, 3), ('Python Guide', 2000, 3),
    ('Стул', 4000, 4), ('Стол', 12000, 4);

INSERT INTO Orders (product_id, quantity, order_date) VALUES 
    (1, 2, '2023-10-01'), (3, 5, '2023-10-02'), (2, 1, '2023-10-03'),
    (5, 3, '2023-10-04'), (1, 1, '2023-10-05'), (9, 4, '2023-10-06');
''')

def run_query(query):
    return pd.read_sql_query(query, conn)

print("База данных для дополнения готова!")

База данных для дополнения готова!


In [83]:
#print("--- ТАБЛИЦА КАТЕГОРИЙ ---")
display(run_query("SELECT * FROM Categories"))

#print("\n--- ТАБЛИЦА ТОВАРОВ ---")
display(run_query("SELECT * FROM Products"))

#print("\n--- ТАБЛИЦА ЗАКАЗОВ ---")
display(run_query("SELECT * FROM Orders"))



,id,name
0,1,Электроника
1,2,Одежда
2,3,Книги
3,4,Мебель


,id,name,price,category_id
0,1,Смартфон,50000.0,1
1,2,Ноутбук,80000.0,1
2,3,Телевизор,45000.0,1
3,4,Футболка,1500.0,2
4,5,Джинсы,3500.0,2
5,6,Куртка,7000.0,2
6,7,SQL для начинающих,1200.0,3
7,8,Python Guide,2000.0,3
8,9,Стул,4000.0,4
9,10,Стол,12000.0,4


,id,product_id,quantity,order_date
0,1,1,2,2023-10-01
1,2,3,5,2023-10-02
2,3,2,1,2023-10-03
3,4,5,3,2023-10-04
4,5,1,1,2023-10-05
5,6,9,4,2023-10-06


### 1. DISTINCT

Команда `DISTINCT` используется для удаления дубликатов. Важно помнить: она всегда применяется ко **всей строке** результата.

#### А) Уникальные значения в одной колонке

In [60]:
# Выбрать только уникальные ID категорий из таблицы товаров
query = "SELECT DISTINCT category_id FROM Products"
run_query(query)

,category_id
0,1
1,2
2,3
3,4


#### Б) Уникальные комбинации (несколько колонок)
Если указать несколько колонок, `DISTINCT` вернет уникальные пары/тройки значений.

In [61]:
# Выбрать уникальные сочетания имени и цены
query = "SELECT DISTINCT name, price FROM Products"
run_query(query)

,name,price
0,Смартфон,50000.0
1,Ноутбук,80000.0
2,Телевизор,45000.0
3,Футболка,1500.0
4,Джинсы,3500.0
5,Куртка,7000.0
6,SQL для начинающих,1200.0
7,Python Guide,2000.0
8,Стул,4000.0
9,Стол,12000.0


#### В) Альтернатива: GROUP BY
Если вам нужно получить уникальные имена, но при этом выполнить какие-то действия с другими колонками (например, найти максимальную цену для каждого имени), лучше использовать `GROUP BY`.

In [62]:
# Группируем по имени и выводим максимальную цену для каждого
query = """
SELECT name, MAX(price) as max_price, category_id 
FROM Products 
GROUP BY name
"""
run_query(query)

,name,max_price,category_id
0,Python Guide,2000.0,3
1,SQL для начинающих,1200.0,3
2,Джинсы,3500.0,2
3,Куртка,7000.0,2
4,Ноутбук,80000.0,1
5,Смартфон,50000.0,1
6,Стол,12000.0,4
7,Стул,4000.0,4
8,Телевизор,45000.0,1
9,Футболка,1500.0,2


### 2. LIKE

Оператор `LIKE` ищет совпадения по тексту с помощью спецсимволов:
- `%` — любое количество любых символов.
- `_` — ровно один любой символ.

#### А) Поиск по количеству символов
Используя несколько `_`, можно задать точную длину или позицию.

In [63]:
# Найти товары, названия которых состоят ровно из 4 букв
# (В нашей базе это 'Стол' и 'Стул')
query = "SELECT * FROM Products WHERE name LIKE '____'"
run_query(query)

,id,name,price,category_id
0,9,Стул,4000.0,4
1,10,Стол,12000.0,4


In [64]:
# Найти товары, названия которых состоят ровно из 1 букв
query = "SELECT SUBSTR(name, 1, 1) as symbol, * FROM Products WHERE name LIKE '_%'"
run_query(query)

,symbol,id,name,price,category_id
0,С,1,Смартфон,50000.0,1
1,Н,2,Ноутбук,80000.0,1
2,Т,3,Телевизор,45000.0,1
3,Ф,4,Футболка,1500.0,2
4,Д,5,Джинсы,3500.0,2
5,К,6,Куртка,7000.0,2
6,S,7,SQL для начинающих,1200.0,3
7,P,8,Python Guide,2000.0,3
8,С,9,Стул,4000.0,4
9,С,10,Стол,12000.0,4


#### Б) Поиск по числам
Хотя `LIKE` текстовый оператор, SQL может применять его и к числам, превращая их в текст.

In [65]:
# Найти цены, которые начинаются на цифру '4'
query = "SELECT * FROM Products WHERE price LIKE '4%'"
run_query(query)

,id,name,price,category_id
0,3,Телевизор,45000.0,1
1,9,Стул,4000.0,4


#### В) Поиск по нескольким словам
Для поиска по нескольким ключевым словам используйте логические операторы:
- **OR** (ИЛИ) — если подходит хотя бы одно слово.
- **AND** (И) — если в названии должны быть оба слова одновременно.

In [66]:
# Найти товары, содержащие 'Guide' ИЛИ 'SQL'
query = "SELECT * FROM Products WHERE name LIKE '%Guide%' OR name LIKE '%SQL%'"
run_query(query)

,id,name,price,category_id
0,7,SQL для начинающих,1200.0,3
1,8,Python Guide,2000.0,3


### 3. IN и BETWEEN

Эти операторы делают условия `WHERE` более читаемыми и компактными.

#### А) BETWEEN (Диапазоны)
Используется для фильтрации данных внутри определенного промежутка.
- **Важно:** Границы (начало и конец) включаются в результат.
- Можно использовать для чисел, дат и текста.
- Конструкция: `column BETWEEN value1 AND value2`.

In [67]:
# Товары с ценой от 1000 до 5000 (включительно) если хотим еще отсортировать то ORDER BY price ASC по возрастанию
query = "SELECT * FROM Products WHERE price BETWEEN 1000 AND 5000"
run_query(query)

,id,name,price,category_id
0,4,Футболка,1500.0,2
1,5,Джинсы,3500.0,2
2,7,SQL для начинающих,1200.0,3
3,8,Python Guide,2000.0,3
4,9,Стул,4000.0,4


#### Б) IN (Списки)
Позволяет указать список конкретных значений, которым должно соответствовать поле.
- Это намного короче, чем писать несколько условий через `OR`.
- Конструкция: `column IN (val1, val2, ...)`.

In [68]:
# Товары из категорий 1 (Электроника) и 3 (Книги)
query = "SELECT * FROM Products WHERE category_id IN (1, 3) ORDER BY category_id, price"
run_query(query)

,id,name,price,category_id
0,3,Телевизор,45000.0,1
1,1,Смартфон,50000.0,1
2,2,Ноутбук,80000.0,1
3,7,SQL для начинающих,1200.0,3
4,8,Python Guide,2000.0,3


#### В) Отрицание: NOT
Оба оператора отлично работают с `NOT`, если нужно исключить данные.
- `NOT BETWEEN` — найти всё, что не попадает в диапазон.
- `NOT IN` — найти всё, кроме указанных значений.

In [69]:
# Исключить товары с ценой от 1000 до 40000
query = "SELECT * FROM Products WHERE price NOT BETWEEN 1000 AND 40000"
run_query(query)

,id,name,price,category_id
0,1,Смартфон,50000.0,1
1,2,Ноутбук,80000.0,1
2,3,Телевизор,45000.0,1


## Задачи

### Задача 5
Выведите все товары, названия которых содержат слово 'Guide'.

In [70]:
# Ваше решение задачи 5
query = "SELECT * FROM Products WHERE name LIKE '%Guide%'"
run_query(query)

,id,name,price,category_id
0,8,Python Guide,2000.0,3


### Задача 6
Найдите все уникальные даты заказов.

In [71]:
# Ваше решение задачи 6
query = "SELECT DISTINCT order_date FROM Orders "
run_query(query)

,order_date
0,2023-10-01
1,2023-10-02
2,2023-10-03
3,2023-10-04
4,2023-10-05
5,2023-10-06


### Задача 7
Товары с ценами: 1500, 4000, 80000.

In [72]:
# Ваше решение задачи 7
query = "SELECT * FROM Products WHERE price IN (1500, 4000, 80000)"
run_query(query)

,id,name,price,category_id
0,2,Ноутбук,80000.0,1
1,4,Футболка,1500.0,2
2,9,Стул,4000.0,4


### Задача 8
Общее количество проданных штук для каждой категории (Название категории | Сумма штук).

In [81]:
# Ваше решение задачи 8
query = """
SELECT 
    c.name AS category_name, 
    SUM(o.quantity) AS total_quantity
FROM Categories c
JOIN Products p ON c.id = p.category_id
LEFT JOIN Orders o ON p.id = o.product_id
GROUP BY c.name

"""
run_query(query)

,category_name,total_quantity
0,Книги,NaN
1,Мебель,4.0
2,Одежда,3.0
3,Электроника,9.0
